In [ ]:
# Linalg libraries
import numpy as np

# Plotting libraries
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Data management libraries
import h5py as hf
from dataclasses import dataclass
import glob

In [ ]:
@dataclass
class ModelTrajectory:
    # Model data
    width: int
    depth: int
    activation: str
    lr: float
    architecture: str
    ds_size: float

    # Loss data
    train_loss: np.ndarray
    test_loss: np.ndarray

    # Accuracy data
    train_accuracy: np.ndarray
    test_accuracy: np.ndarray

    # CV data
    train_entropy: np.ndarray
    train_trace: np.ndarray
    test_entropy: np.ndarray
    test_trace: np.ndarray


In [ ]:
def load_model_trajectory(file_root, seed: int):
    # Glob all files with the correct signature
    files = glob.glob(f"{file_root}/*{seed}*.h5")

    # Extract model parameters
    model_params = files[0].split("/")[-1].split('_')
    width = int(model_params[2])
    depth = int(model_params[3])
    activation = model_params[4]
    ds_size = float(model_params[5])    
    lr = float(model_params[7])
    architecture = model_params[1]

    for file in files:
        with hf.File(file, 'r') as f:
            if file.split("/")[-1].split("_")[0] == "train":
                train_loss = f['loss'][:]
                train_accuracy = f['accuracy'][:]
                train_entropy = f['entropy'][:]
                train_trace = f['trace'][:]
            else:
                test_loss = f['loss'][:]
                test_accuracy = f['accuracy'][:]
                test_entropy = f['entropy'][:]
                test_trace = f['trace'][:]

    return ModelTrajectory(
        width=width,
        depth=depth,
        activation=activation,
        lr=lr,
        architecture=architecture,
        ds_size=ds_size,
        train_loss=train_loss,
        test_loss=test_loss,
        train_entropy=train_entropy,
        train_trace=train_trace,
        test_entropy=test_entropy,
        test_trace=test_trace,
        train_accuracy=train_accuracy,
        test_accuracy=test_accuracy,
    )

def load_model_trajectories(root_path):
    files = glob.glob(root_path + '/*.h5')

    # Get unique names
    unique_names = []
    for file in files:
        unique_names.append(file.split('/')[-1].split('_')[-2])

    unique = np.unique(unique_names)
    # Load data
    data = []
    for name in unique:
        data.append(load_model_trajectory(root_path, name))

    return data   

In [ ]:
# fuel_mse_order_two = load_model_trajectories("./mse-2")
# fuel_mse_order_four = load_model_trajectories("./mse-4")
data = load_model_trajectories("./adam-slow")

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 15))

line_styles = ['-', '--']
train_colours = ['#101419', '#30C5FF']
test_colours = ['#09BC8A', '#F7E733']

# Activation function scaling
widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]
widths=[50]
depths=[3]
activations=["relu", "tanh"]
row = 0
for width in widths:
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                if model.activation in activations:      
                    ax[row][0].scatter(np.log(model.train_loss), model.train_entropy,  s=1, c=train_colours[activations.index(model.activation)], label=model.activation)
                    ax[row][0].scatter(np.log(model.test_loss), model.test_entropy, marker='x',  s=1, c=test_colours[activations.index(model.activation)], label=model.activation)

                    ax[row][1].scatter(np.log(model.train_loss), np.log(model.train_trace), s=1, c=train_colours[activations.index(model.activation)])
                    ax[row][1].scatter(np.log(model.test_loss), np.log(model.test_trace), marker='x', s=1, c=test_colours[activations.index(model.activation)])

                    ax[row][2].plot(model.train_loss, '.', color=train_colours[activations.index(model.activation)], markersize=1)
                    ax[row][2].plot(model.test_loss, 'x', color=test_colours[activations.index(model.activation)], markersize=1)

                    # ax[row][3].plot(model.train_entropy, '.', color=colours[activations.index(model.activation)], markersize=1)
                    # ax[row][3].plot(model.test_entropy, 'x', color=colours[activations.index(model.activation)], markersize=1)

                    # ax[row][4].plot(model.train_trace, '.', color=colours[activations.index(model.activation)], markersize=1)
                    # ax[row][4].plot(model.test_trace, 'x', color=colours[activations.index(model.activation)], markersize=1)

ax[row][0].set_ylabel("S")
ax[row][0].set_xlabel("log(Train Loss)")
ax[row][0].invert_xaxis()

blue_patch = mpatches.Patch(color='#101419', label='Train ReLU')
red_patch = mpatches.Patch(color='#30C5FF', label='Train Tanh')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test ReLU')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test Tanh')
ax[row][0].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Activation")

ax[row][1].set_xlabel("log(Train Loss)")
ax[row][1].set_ylabel("log(Trace)")
ax[row][1].invert_xaxis()
ax[row][1].set_title("Activation Scaling (50, 3, x)")

ax[row][2].set_xlabel("Epoch")
ax[row][2].set_ylabel("Loss")
ax[row][2].set_yscale("log")

# ax[row][3].set_ylabel("Entropy")
# ax[row][3].set_xlabel("Epoch")

# ax[row][4].set_ylabel("Trace")
# ax[row][4].set_xlabel("Epoch")

# Width scaling
widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]
widths=[12, 1000]
depths=[3]
activations=["relu"]

row = 1
for width in widths:
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                if model.activation in activations:      
                    ax[row][0].scatter(np.log(model.train_loss), model.train_entropy,  s=1, c=train_colours[widths.index(model.width)])
                    ax[row][0].scatter(np.log(model.test_loss), model.test_entropy, marker='x',  s=1, c=test_colours[widths.index(model.width)])

                    ax[row][1].scatter(np.log(model.train_loss), np.log(model.train_trace), s=1, c=train_colours[widths.index(model.width)])
                    ax[row][1].scatter(np.log(model.test_loss), np.log(model.test_trace), marker='x', s=1, c=test_colours[widths.index(model.width)])

                    ax[row][2].plot(model.train_loss, '.', color=train_colours[widths.index(model.width)], markersize=1)
                    ax[row][2].plot(model.test_loss, 'x', color=test_colours[widths.index(model.width)], markersize=1)

                    # ax[row][3].plot(model.train_entropy, '.', color=colours[widths.index(model.width)], markersize=1)
                    # ax[row][3].plot(model.test_entropy, 'x', color=colours[widths.index(model.width)], markersize=1)

                    # ax[row][4].plot(model.train_trace, '.', color=colours[widths.index(model.width)], markersize=1)
                    # ax[row][4].plot(model.test_trace, 'x', color=colours[widths.index(model.width)], markersize=1)

ax[row][0].set_ylabel("S")
ax[row][0].set_xlabel("log(Train Loss)")
ax[row][0].invert_xaxis()

blue_patch = mpatches.Patch(color='#101419', label='Train 12')
red_patch = mpatches.Patch(color='#30C5FF', label='Train 1000')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test 12')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test 1000')
ax[row][0].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Width")


ax[row][1].set_xlabel("log(Train Loss)")
ax[row][1].set_ylabel("log(Trace)")
ax[row][1].invert_xaxis()
ax[row][1].set_title("Width Scaling (x, 3, ReLU)")

ax[row][2].set_xlabel("Epoch")
ax[row][2].set_ylabel("Loss")
ax[row][2].set_yscale("log")

# ax[row][3].set_ylabel("Entropy")
# ax[row][3].set_xlabel("Epoch")

# ax[row][4].set_ylabel("Trace")
# ax[row][4].set_xlabel("Epoch")

# Depth scaling
widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]
widths=[500]
depths=[2, 10]
activations=["relu"]

row = 2
for width in widths:
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                if model.activation in activations:      
                    ax[row][0].scatter(np.log(model.train_loss), model.train_entropy,  s=1, c=train_colours[depths.index(model.depth)], label=model.depth)
                    ax[row][0].scatter(np.log(model.test_loss), model.test_entropy, marker='x',  s=1, c=test_colours[depths.index(model.depth)], label=model.depth)

                    ax[row][1].scatter(np.log(model.train_loss), np.log(model.train_trace), s=1, c=train_colours[depths.index(model.depth)])
                    ax[row][1].scatter(np.log(model.test_loss), np.log(model.test_trace), marker='x', s=1, c=test_colours[depths.index(model.depth)])

                    ax[row][2].plot(model.train_loss, '.', color=train_colours[depths.index(model.depth)], markersize=1)
                    ax[row][2].plot(model.test_loss, 'x', color=test_colours[depths.index(model.depth)], markersize=1)

                    # ax[row][3].plot(model.train_entropy, '.', color=colours[depths.index(model.depth)], markersize=1)
                    # ax[row][3].plot(model.test_entropy, 'x', color=colours[depths.index(model.depth)], markersize=1)

                    # ax[row][4].plot(model.train_trace, '.', color=colours[depths.index(model.depth)], markersize=1)
                    # ax[row][4].plot(model.test_trace, 'x', color=colours[depths.index(model.depth)], markersize=1)

ax[row][0].set_ylabel("S")
ax[row][0].set_xlabel("log(Train Loss)")
ax[row][0].invert_xaxis()

blue_patch = mpatches.Patch(color='#101419', label='Train 2')
red_patch = mpatches.Patch(color='#30C5FF', label='Train 10')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test 2')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test 10')
ax[row][0].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Depth")

ax[row][1].set_xlabel("log(Train Loss)")
ax[row][1].set_ylabel("log(Trace)")
ax[row][1].invert_xaxis()
ax[row][1].set_title("Depth Scaling (500, x, ReLU)")

ax[row][2].set_xlabel("Epoch")
ax[row][2].set_ylabel("Loss")
ax[row][2].set_yscale("log")

# ax[row][3].set_ylabel("Entropy")
# ax[row][3].set_xlabel("Epoch")

# ax[row][4].set_ylabel("Trace")
# ax[row][4].set_xlabel("Epoch")

plt.savefig("mnist-dense-phases.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 15))

line_styles = ['-', '--']
train_colours = ['#101419', '#30C5FF']
test_colours = ['#09BC8A', '#F7E733']

# Activation function scaling
widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]
widths=[50]
depths=[3]
activations=["relu", "tanh"]

row = 0
for width in widths:
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                if model.activation in activations:      
                    ax[row][0].plot(model.train_entropy,  ".", color=train_colours[activations.index(model.activation)], label=model.activation, markersize=1)
                    ax[row][0].plot(model.test_entropy, ".", color=test_colours[activations.index(model.activation)], label=model.activation, markersize=1)

                    ax[row][1].plot(np.log(model.train_trace), ".", color=train_colours[activations.index(model.activation)], markersize=1)
                    ax[row][1].plot(np.log(model.test_trace), ".", color=test_colours[activations.index(model.activation)], markersize=1)

                    ax[row][2].plot(model.train_loss, '.', color=train_colours[activations.index(model.activation)], markersize=1)
                    ax[row][2].plot(model.test_loss, 'x', color=test_colours[activations.index(model.activation)], markersize=1)

                    # ax[row][3].plot(model.train_entropy, '.', color=colours[activations.index(model.activation)], markersize=1)
                    # ax[row][3].plot(model.test_entropy, 'x', color=colours[activations.index(model.activation)], markersize=1)

                    # ax[row][4].plot(model.train_trace, '.', color=colours[activations.index(model.activation)], markersize=1)
                    # ax[row][4].plot(model.test_trace, 'x', color=colours[activations.index(model.activation)], markersize=1)

ax[row][0].set_ylabel("S")
ax[row][0].set_xlabel("Epoch")
# ax[row][0].invert_xaxis()

blue_patch = mpatches.Patch(color='#101419', label='Train ReLU')
red_patch = mpatches.Patch(color='#30C5FF', label='Train Tanh')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test ReLU')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test Tanh')
ax[row][0].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Activation")

ax[row][1].set_xlabel("Epoch")
ax[row][1].set_ylabel("log(Trace)")
# ax[row][1].invert_xaxis()
ax[row][1].set_title("Activation Scaling (32, 2, x)")

ax[row][2].set_xlabel("Epoch")
ax[row][2].set_ylabel("Loss")
ax[row][2].set_yscale("log")

# ax[row][3].set_ylabel("Entropy")
# ax[row][3].set_xlabel("Epoch")

# ax[row][4].set_ylabel("Trace")
# ax[row][4].set_xlabel("Epoch")

# Width scaling
widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]
widths=[12, 1000]
depths=[3]
activations=["relu"]

row = 1
for width in widths:
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                if model.activation in activations:      
                    ax[row][0].plot(model.train_entropy, ".", color=train_colours[widths.index(model.width)], markersize=1)
                    ax[row][0].plot(model.test_entropy, ".", color=test_colours[widths.index(model.width)], markersize=1)

                    ax[row][1].plot(np.log(model.train_trace), ".", color=train_colours[widths.index(model.width)], markersize=1)
                    ax[row][1].plot(np.log(model.test_trace), ".", color=test_colours[widths.index(model.width)], markersize=1)

                    ax[row][2].plot(model.train_loss, '.', color=train_colours[widths.index(model.width)], markersize=1)
                    ax[row][2].plot(model.test_loss, 'x', color=test_colours[widths.index(model.width)], markersize=1)

                    # ax[row][3].plot(model.train_entropy, '.', color=colours[widths.index(model.width)], markersize=1)
                    # ax[row][3].plot(model.test_entropy, 'x', color=colours[widths.index(model.width)], markersize=1)

                    # ax[row][4].plot(model.train_trace, '.', color=colours[widths.index(model.width)], markersize=1)
                    # ax[row][4].plot(model.test_trace, 'x', color=colours[widths.index(model.width)], markersize=1)

ax[row][0].set_ylabel("S")
ax[row][0].set_xlabel("Epoch")
# ax[row][0].invert_xaxis()

blue_patch = mpatches.Patch(color='#101419', label='Train 12')
red_patch = mpatches.Patch(color='#30C5FF', label='Train 1000')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test 12')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test 1000')
ax[row][0].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Width")


ax[row][1].set_xlabel("Epoch")
ax[row][1].set_ylabel("log(Trace)")
# ax[row][1].invert_xaxis()
ax[row][1].set_title("Width Scaling (x, 3, ReLU)")

ax[row][2].set_xlabel("Epoch")
ax[row][2].set_ylabel("Loss")
ax[row][2].set_yscale("log")

# ax[row][3].set_ylabel("Entropy")
# ax[row][3].set_xlabel("Epoch")

# ax[row][4].set_ylabel("Trace")
# ax[row][4].set_xlabel("Epoch")

# Depth scaling
widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]
widths=[500]
depths=[2, 5]
activations=["tanh"]

row = 2
for width in widths:
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                if model.activation in activations:      
                    ax[row][0].plot(model.train_entropy,  ".", color=train_colours[depths.index(model.depth)], label=model.depth, markersize=1)
                    ax[row][0].plot(model.test_entropy, ".", color=test_colours[depths.index(model.depth)], label=model.depth, markersize=1)

                    ax[row][1].plot(np.log(model.train_trace), ".", color=train_colours[depths.index(model.depth)], markersize=1)
                    ax[row][1].plot(np.log(model.test_trace), ".", color=test_colours[depths.index(model.depth)], markersize=1)

                    ax[row][2].plot(model.train_loss, '.', color=train_colours[depths.index(model.depth)], markersize=1)
                    ax[row][2].plot(model.test_loss, 'x', color=test_colours[depths.index(model.depth)], markersize=1)

                    # ax[row][3].plot(model.train_entropy, '.', color=colours[depths.index(model.depth)], markersize=1)
                    # ax[row][3].plot(model.test_entropy, 'x', color=colours[depths.index(model.depth)], markersize=1)

                    # ax[row][4].plot(model.train_trace, '.', color=colours[depths.index(model.depth)], markersize=1)
                    # ax[row][4].plot(model.test_trace, 'x', color=colours[depths.index(model.depth)], markersize=1)

ax[row][0].set_ylabel("S")
ax[row][0].set_xlabel("Epoch)")
# ax[row][0].invert_xaxis()

blue_patch = mpatches.Patch(color='#101419', label='Train 2')
red_patch = mpatches.Patch(color='#30C5FF', label='Train 10')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test 2')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test 10')
ax[row][0].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Depth")

ax[row][1].set_xlabel("Epoch")
ax[row][1].set_ylabel("log(Trace)")
# ax[row][1].invert_xaxis()
ax[row][1].set_title("Depth Scaling (64, x, ReLU)")

ax[row][2].set_xlabel("Epoch")
ax[row][2].set_ylabel("Loss")
ax[row][2].set_yscale("log")
ax[row][0].set_xscale("log")
ax[row][1].set_xscale("log")



# ax[row][3].set_ylabel("Entropy")
# ax[row][3].set_xlabel("Epoch")

# ax[row][4].set_ylabel("Trace")
# ax[row][4].set_xlabel("Epoch")

# plt.savefig("mnist-dense-evolution.pdf")
plt.show()